# 03 · Aerial on-road and off-road greenery per LSOA and ward
Splits each area's green (total from 01) into three parts that add up exactly: **on-road** (inside the road strips from 02; junction overlaps count once, roads are cut at area boundaries), **accessible off-road** (in public parks and gardens) and **inaccessible off-road** (all other green).

| Output column | Name in the analysis tables | Meaning |
|---|---|---|
| `on_road_green_share` / `_weighted` | `SAT_on_road_share_u` / `_w` | green ÷ road-strip area (unweighted / weighted by road centrality) |
| `accessible_off_road_green` | `SAT_off_road` | green in public parks and gardens ÷ area |

**Inputs:** `SAT_greenery_per_segment_tiles/road_buffers.gpkg`, `SAT_greenery_per_segment.csv`, `SAT_greenery_total_{lsoa,ward}.csv`, aerial tiles (not included), `data/raw_data/ward_public_green/`, `data/raw_data/M25/` (road centrality).
**Outputs:** `data/greenery_metrics/SAT_greenery_on_off_road_{lsoa,ward}.csv`, `SAT_greenery_off_road_accessibility_{lsoa,ward}.csv`; per-tile cache in `SAT_greenery_on_off_road_tiles/`.
**Runtime:** about 1 hour on 8 cores. **Logic:** `03_SAT_greenery_on_off_road_workers.py`.

In [ ]:
import importlib
import os
import sys
from pathlib import Path

from joblib import Parallel, delayed

sys.path.insert(0, str(Path.cwd().resolve()))
oow = importlib.import_module('03_SAT_greenery_on_off_road_workers')

## 1. On-road and accessible green per tile, LSOA and ward

In [ ]:
N_JOBS = os.cpu_count()   # lower if memory is tight
tiles = oow._seg.tile_paths()
todo = [t for t in tiles if not oow.tile_done(t)]
print(f'{len(tiles) - len(todo)}/{len(tiles)} tiles done; running {len(todo)}')
Parallel(n_jobs=N_JOBS, verbose=5)(delayed(oow.process_tile)(t) for t in todo);

## 2. Combine the tiles → SAVE

In [ ]:
results = {label: oow.merge_tiles(label) for label in oow.CONFIGS}
for label, df in results.items():
    for path in oow.write_csvs(df, label):
        print(f'wrote {len(df):,} {label}s → {path.name}')
results['ward'].head()

## Validation

In [ ]:
FRACS = ['on_road_green_area', 'accessible_off_road_green', 'inaccessible_off_road_green', 'off_road_green_area', 'total_green_area']
for label, df in results.items():
    assert df[oow.CONFIGS[label].area_id_col].is_unique and len(df) > 0
    assert df[FRACS].apply(lambda s: s.between(0, 1)).all().all()
    parts = df['on_road_green_area'] + df['accessible_off_road_green'] + df['inaccessible_off_road_green']
    assert ((parts - df['total_green_area']).abs() < 1e-3).all()
    assert df['on_road_green_share'].dropna().between(0, 1 + 1e-3).all()
    assert df['on_road_green_share_weighted'].dropna().between(0, 1).all()
    print(f'[{label}] {len(df):,} areas | means:', df[FRACS].mean().round(4).to_dict())